## Part 1: class summary by buffer

**Calculate total area by class for buffers of CWMP and GLFWRA bird count locations.**

_Buffers_: 200m, 500m, 1km, 2km  
_Data_: land cover & wetland type  
_Years_: 2010, 2015, 2020, 2025  
_Deliverable_: CSV file containing total area for each class for each buffer for each location.


In [1]:
# 0) RESTART the Jupyter kernel first (not just "Run All").
# 1) Run this as the very first cell:
import os, sys

# Drop any OSGeo4W bleed-through
for k in ["PROJ_LIB", "PROJ_DATA", "GDAL_DATA"]:
    if os.environ.get(k, "").lower().find("osgeo4w") != -1:
        os.environ.pop(k, None)
os.environ["PATH"] = ";".join(
    p for p in os.environ["PATH"].split(";") if "OSGeo4W" not in p
)

from pyproj import CRS, datadir

crs4326 = CRS.from_epsg(4326)
print("CRS ok:", crs4326)
print("pyproj datadir:", datadir.get_data_dir())
print("PROJ_LIB:", os.environ.get("PROJ_LIB"))

CRS ok: EPSG:4326
pyproj datadir: C:\Users\kristofj\Miniconda3\envs\harvest-patch\Library\share\proj
PROJ_LIB: C:\Users\kristofj\Miniconda3\envs\harvest-patch\Library\share\proj


## Install necessary libraries

### Subtask:

Install libraries like `boto3` for interacting with AWS S3 and `rasterio` or `rioxarray` for reading the GeoTIFF data.


**Reasoning**:
Install the necessary libraries for interacting with AWS S3 and reading GeoTIFF data.


In [2]:
import boto3, os, rasterio, shapely, re, requests
import pandas as pd, geopandas as gpd, numpy as np
from rasterio.session import AWSSession
from rasterio.mask import mask
from geopandas.array import GeometryArray

## Constants


In [3]:
# paths
cwmp_path = "data/CWMP_PCL_all.xlsx - CWMP_bird_point_count_locations.csv"

# buffer sizes
buffers = [200, 500, 1000, 2000]

# years
years = [2010, 2015, 2020, 2023]

# project CRS
crs = "EPSG:5070"

In [4]:
NLCD_CLASS_LUT = {
    # 11: "Open Water",
    12: "Perennial Ice/Snow",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land (Rock/Sand/Clay)",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
    # Optional (present in metadata legend):
    250: "NoData",
}

NALCMS_CLASS_LUT = {
    1: "sub-polar needleleaf forest",
    2: "Sub-polar taiga needleleaf forest",
    3: "sub-tropical broadleaf evergreen forest",
    4: "sub-tropical broadleaf deciduous forest",
    5: "sub-polar broadleaf deciduous forest",
    6: "Mixed Forest",
    7: "sub-tropical shrubland",
    8: "sub-polar shrubland",
    9: "sub-tropical grassland",
    10: "sub-polar grassland",
    11: "polar shrubland-lichen-moss",
    12: "polar grassland-lichen-moss",
    13: "polar barren-lichen-moss",
    14: "Wetland",
    15: "Cropland",
    16: "Barren lands",
    17: "Urban",
    # 18: "Water",
    19: "Snow and Ice",
}

## Configure Rasterio for Anonymous S3 Access


In [5]:
%pip install python-dotenv
# AWS credentials
from pathlib import Path
import os

# auto-load .env if present
try:
    from dotenv import load_dotenv

    # Point to the same path as VS Code or wherever you placed it
    load_dotenv(dotenv_path=Path.cwd() / ".env")
except Exception:
    pass

# set the default region
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
# set no sign in request
os.environ["AWS_NO_SIGN_REQUEST"] = "NO"

Note: you may need to restart the kernel to use updated packages.


## NLCD Access


In [6]:
import boto3

# Specify the bucket name and prefix
bucket_name = "usgs-landcover"
base_prefix = "annual-nlcd/c1/v0/cu/mosaic/"

# Define the years of interest and their expected filenames
nlcd_filenames = {
    2010: "Annual_NLCD_LndCov_2010_CU_C1V0.tif",
    2015: "Annual_NLCD_LndCov_2015_CU_C1V0.tif",
    2020: "Annual_NLCD_LndCov_2020_CU_C1V0.tif",
    2023: "Annual_NLCD_LndCov_2023_CU_C1V0.tif",  # Assuming similar naming convention for 2023
}

# Initialize the S3 client
# Using default configuration, relying on AWS environment variables or public bucket policy for access
s3 = boto3.client("s3", region_name="us-west-2")

print("Verifying existence of NLCD land cover files in S3:")

# Check for the existence of each file
found_objects = False
nlcd_paths = {}
for year, filename in nlcd_filenames.items():
    s3_key = base_prefix + filename
    try:
        # Use head_object to check for file existence without downloading
        # Add RequestPayer='requester' as it's a requester pays bucket
        response = s3.head_object(
            Bucket=bucket_name, Key=s3_key, RequestPayer="requester"
        )
        print(f"File found for {year}: s3://{bucket_name}/{s3_key}")
        # print(response)
        # add to nlcd_path dict
        nlcd_paths[year] = f"s3://{bucket_name}/{s3_key}"
        found_objects = True
    except s3.exceptions.ClientError as e:
        if e.response["Error"]["Code"] == "404":
            print(f"File NOT found for {year}: s3://{bucket_name}/{s3_key}")
        else:
            print(f"An error occurred checking file for {year}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred for {year}: {e}")

if not found_objects:
    print(
        "No requested NLCD land cover files were found in the specified location."
    )

Verifying existence of NLCD land cover files in S3:
File found for 2010: s3://usgs-landcover/annual-nlcd/c1/v0/cu/mosaic/Annual_NLCD_LndCov_2010_CU_C1V0.tif
File found for 2015: s3://usgs-landcover/annual-nlcd/c1/v0/cu/mosaic/Annual_NLCD_LndCov_2015_CU_C1V0.tif
File found for 2020: s3://usgs-landcover/annual-nlcd/c1/v0/cu/mosaic/Annual_NLCD_LndCov_2020_CU_C1V0.tif
File found for 2023: s3://usgs-landcover/annual-nlcd/c1/v0/cu/mosaic/Annual_NLCD_LndCov_2023_CU_C1V0.tif


Initialize boto3 and AWS sessions


In [7]:
# build a signed AWS session that includes requester-pays
boto_sesh = boto3.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name="us-west-2",
)
aws_sesh = AWSSession(boto_sesh, requester_pays=True)

## Load buffer data

bird count locations:

- [coastal wetlands](https://docs.google.com/spreadsheets/d/1-FZrtUYZS9HEFVKIO94fKBhGXi5KQDon)
  - field mapping
    - lat: latitude
    - lon: longitude
    - id: `f"siteid_waypoint"`
- GLFWRA


In [8]:
# load cwmp table
cwmp_df = pd.read_csv(cwmp_path)
print(f"CWMP bird count locations: {len(cwmp_df)}")
cwmp_df.head()

CWMP bird count locations: 4941


,site_id,crew_name,year,date,sample,point_id,bearing,waypoint,longitude,latitude,coord_qual,comments
0,1,SUNY-B,2015,5/28/2015,1,1,140,AB0001.1.BE,-78.840043,43.312097,Excellent,NaN
1,1,SUNY-B,2020,6/1/2020,1,1,238,AB0001.1.AM,-78.840627,43.312285,Confirmed,NaN
2,4,SUNY-B,2015,5/28/2015,1,1,350,AB0004.1.BE,-78.744200,43.329043,Excellent,NaN
3,5,SUNY-B,2013,6/12/2013,1,1,183,B0005.PM,-78.715802,43.334753,Excellent,NaN
4,5,SUNY-B,2018,5/24/2018,1,1,180,AB0005.1.PM,-78.716118,43.335103,Excellent,NaN


### Helper Functions


In [9]:
# function to transform nice string into a name
# replacing spaces and other non-alphanumeric characters with underscores
def nice_string_to_name(nice_string: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]+", "_", nice_string.replace(" ", "_")).lower()

## Process each location, buffer

1. create buffer
2. read in raster for buffer
3. calculate stats for buffer

- for each class:
  - count
  - pct of total


### NLCD Processing


In [10]:
def nlcd_buffer_clip(
    point: GeometryArray, buffer_size: int, year: int
) -> tuple[np.ndarray, float]:
    # get raster values for buffer
    with rasterio.Env(aws_sesh):
        with rasterio.open(nlcd_paths[year]) as src:
            # Calculate area per pixel in square meters
            area_per_pixel = abs(src.res[0] * src.res[1])

            # reproject point to raster CRS - so units match buffer units
            point = point.to_crs(src.crs)
            # create buffer from point, convert to geojson like object
            buffer = point.buffer(buffer_size)

            # read data from raster within poly (window)
            data, transform = mask(
                src, buffer, crop=True, filled=True
            )  # windowed read
            # get values from first band, skipping nodata values
            band = data[0]  # NLCD is single-band categorical
            nodata = src.nodatavals[0]
            mask_valid = (
                (band != nodata)
                if nodata is not None
                else np.ones_like(band, dtype=bool)
            )
            return band[mask_valid].astype(np.int8), area_per_pixel

### NA Land Cover

In [11]:
nalcms_paths = {
    2010: r"M:\isabel_bird\data\nalcms\land_cover_2010v3_30m_tif\NA_NALCMS_landcover_2010v3_30m\data\NA_NALCMS_landcover_2010v3_30m.tif",
    2015: r"M:\isabel_bird\data\nalcms\land_cover_2015v4_30m_tif\NA_NALCMS_landcover_2015v4_30m\data\NA_NALCMS_landcover_2015v4_30m.tif",
    2020: r"M:\isabel_bird\data\nalcms\land_cover_2020v2_30m_tif\NA_NALCMS_landcover_2020v2_30m\data\NA_NALCMS_landcover_2020v2_30m.tif",
}


def nalcms_buffer_clip(
    point: GeometryArray, buffer_size: int, year: int
) -> tuple[np.ndarray, float]:
    # get raster values for buffer
    with rasterio.open(nalcms_paths[year]) as src:
        # Calculate area per pixel in square meters
        area_per_pixel = abs(src.res[0] * src.res[1])

        # reproject point to raster CRS - so units match buffer units
        point = point.to_crs(src.crs)
        # create buffer from point, convert to geojson like object
        buffer = point.buffer(buffer_size)

        # read data from raster within poly (window)
        data, transform = mask(
            src, buffer, crop=True, filled=True
        )  # windowed read
        # get values from first band, skipping nodata values
        band = data[0]  # NLCD is single-band categorical
        nodata = src.nodatavals[0]
        mask_valid = (
            (band != nodata)
            if nodata is not None
            else np.ones_like(band, dtype=bool)
        )
        return band[mask_valid].astype(np.int8), area_per_pixel

### Land Cover - One Row

In [12]:
def process_one_landcover(
    point: GeometryArray, buffer_size: int, year: int, prefix: str
) -> dict:
    # Initialize empty result dict
    result = {}

    try:
        # get raster values for buffer
        if prefix == "nlcd":
            if year not in nlcd_paths:
                return result
            vals, area_per_pixel = nlcd_buffer_clip(point, buffer_size, year)
        else:
            if year not in nalcms_paths:
                return result
            vals, area_per_pixel = nalcms_buffer_clip(point, buffer_size, year)

        # Check if we got any valid values
        if vals is None or len(vals) == 0:
            return result

        # get count by class
        unique_values, counts = np.unique(vals, return_counts=True)
        class_counts = dict(zip(unique_values, counts))

        # calculate percent of total area occupied by each class
        total = np.sum(counts)
        if total == 0:
            return result

        class_percents = {k: v / total for k, v in class_counts.items()}

        # Use appropriate lookup table
        lut = NLCD_CLASS_LUT if prefix == "nlcd" else NALCMS_CLASS_LUT

        # iterate over unique values and update dict with count, percent
        for val in [v for v in unique_values if v in lut.keys()]:
            result.update(
                {
                    f"{prefix}_buff{buffer_size}_yr{year}_{nice_string_to_name(lut[val])}_count": class_counts[
                        val
                    ],
                    f"{prefix}_buff{buffer_size}_yr{year}_{nice_string_to_name(lut[val])}_area": class_counts[
                        val
                    ]
                    * area_per_pixel,
                    f"{prefix}_buff{buffer_size}_yr{year}_{nice_string_to_name(lut[val])}_percent": class_percents[
                        val
                    ],
                }
            )

    except Exception as e:
        print(
            f"Error processing {prefix} data for year {year}, buffer {buffer_size}m: {e}"
        )

    return result

### Wetlands Processing


In [13]:
import geopandas as gpd, pandas as pd, pyogrio as po, shapely
from pathlib import Path

crs = "EPSG: 5070"
nwi_dir = r"M:\isabel_bird\data\nwi"
col_mapping = {
    "WETLAND_TYPE": "type",
    "ATTRIBUTE": "code",
}


def wetlands_within_buffer(buffer: gpd.GeoSeries) -> gpd.GeoDataFrame | None:
    # get geopackages
    gpkg_files = Path(nwi_dir).rglob("*.gpkg")

    # find which files intersect the buffer
    in_gdfs = []
    for gpkg in gpkg_files:
        # print(f"Checking {gpkg}...")
        # layer name will always be state name abbreviation + "_Wetlands"
        layer_name = f"{gpkg.stem.split('_')[0]}_Wetlands"
        # get NWI CRS and bounds
        nwi_info = po.read_info(gpkg, layer=layer_name)
        # reproject buffer to nwi_crs, get bounds
        bbox = tuple(buffer.to_crs(nwi_info["crs"]).total_bounds)
        # check if bbox intersects nwi bounds
        if shapely.box(*bbox).intersects(
            shapely.box(*nwi_info["total_bounds"])
        ):
            # if we're passed bbox intersect, read into geodataframe using bbox
            gdf = po.read_dataframe(gpkg, bbox=bbox, layer=layer_name)
            # if gdf is not empty, add to list
            if gdf is not None and not gdf.empty:
                # print(f"  Found {len(gdf)} wetlands in {gpkg}")
                in_gdfs.append(gdf)

    # exit early if no wetlands found
    if not in_gdfs:
        return None

    # combine all gdfs
    combined_gdf = gpd.GeoDataFrame(
        pd.concat(in_gdfs, ignore_index=True), crs=crs
    )

    # clip combined gdf using buffer
    clipped_gdf = gpd.clip(combined_gdf, buffer.to_crs(crs))
    return clipped_gdf


def process_one_nwi(point: GeometryArray, buffer_size: int, prefix: str):
    # print(f"Processing NWI for buffer size {buffer_size} m")
    # print(f"    Getting bounding box for point...")
    # reproject point to raster CRS - so units match buffer units
    point = point.to_crs(crs)
    # create buffer as GeoSeries from point
    buffer = gpd.GeoSeries(point, crs=crs).buffer(buffer_size)

    # get wetlands within bbox
    # print(f"    Getting wetlands within buffer...")
    gdf = wetlands_within_buffer(buffer)

    # exit early if no wetlands found
    if gdf is None:
        # print("No wetlands found within bounding box.")
        return

    # print(f"    Found {len(gdf)} wetlands within buffer.")

    # print(f"    Generating column statistics...")
    # iterate over columns
    # find unique values and counts of each column in col_mapping
    # and calculate total area and percent of total area occupied by each class
    result = {}

    # calculate total area of all wetlands
    total_area = gdf.geometry.area.sum()

    # iterate over columns in col_mapping
    for col in col_mapping.keys():
        # get count by class
        class_counts = gdf[col].value_counts()
        # calculate area by class
        class_areas = gdf.groupby(col)["geometry"].apply(lambda x: x.area.sum())
        # calculate percent area
        class_pcts = class_areas / total_area

        # add to results
        for val in class_counts.index:
            name = f"{prefix}_buff{buffer_size}_{col_mapping[col]}_{val}"
            result.update(
                {
                    f"{name}_count": class_counts[val],
                    f"{name}_area": class_areas[val],
                    f"{name}_percent": class_pcts[val],
                }
            )

    return result
    # print(f"result: {result}")

## Process One Loop - Plotting Output

Modify the code in the selected cell to create a map showing the buffer outline overlaid on the clipped raster with distinct colors for categorical data, create a dataframe combining the class counts and class percent dictionaries, and create a pie chart plot based on the class count percent data.


### Install necessary libraries

#### Subtask:

Install libraries for plotting and data manipulation, such as `matplotlib` and `seaborn`.


**Reasoning**:
Install libraries for plotting and data manipulation, such as `matplotlib` and `seaborn`.


In [14]:
%pip install matplotlib seaborn
import matplotlib.pyplot as plt
import seaborn as sns

Note: you may need to restart the kernel to use updated packages.


### `process one for show` function

#### Subtask:

Update the function to perform the following tasks:

- Create a map showing the buffer outline overlaid on the clipped raster with distinct colors for categorical data.
- Create a pandas DataFrame combining the class counts and class percent dictionaries.
- Generate a pie chart based on the class percent data.


In [15]:
import rasterio.plot
import rasterio.windows
import math  # Import math for isnan check
import geopandas as gpd  # Import geopandas


def process_one_for_show(row: pd.Series, buffer_size: int, year: int):
    # create point
    point = gpd.points_from_xy(
        [row["longitude"]], [row["latitude"]], crs="EPSG:4326"
    )

    # Check if point creation was successful and coordinates are valid
    if point.is_empty or not all(math.isfinite(c) for c in point[0].coords[0]):
        print(
            f"Skipping location {row['site_id']} due to invalid coordinates: {row['longitude']}, {row['latitude']}"
        )
        return None

    # get raster values for buffer
    with rasterio.Env(aws_sesh):
        with rasterio.open(nlcd_paths[year]) as src:
            # reproject point to raster CRS - so units match buffer units
            point_reproj = point.to_crs(src.crs)
            # create buffer from point, convert to geojson like object
            buffer_reproj = point_reproj.buffer(buffer_size)

            # Check if buffer creation was successful and bounds are valid
            # Iterate over the array within the list returned by buffer_reproj.bounds
            if buffer_reproj.is_empty or (
                buffer_reproj.bounds is not None
                and any(math.isnan(c) for c in buffer_reproj.bounds[0])
            ):
                print(
                    f"Skipping location {row['site_id']} due to invalid buffer geometry. Bounds: {buffer_reproj.bounds}"
                )
                return None

            # Add this check to see what buffer_reproj.bounds contains
            print(f"buffer_reproj.bounds: {buffer_reproj.bounds}")

            # read data from raster within poly (window)
            data, transform = mask(
                src, buffer_reproj, crop=True, filled=True
            )  # windowed read
            # get values from first band, skipping nodata values
            band = data[0]  # NLCD is single-band categorical
            nodata = src.nodatavals[0]
            mask_valid = (
                (band != nodata)
                if nodata is not None
                else np.ones_like(band, dtype=bool)
            )
            vals = band[mask_valid].astype(np.int8)

            # get count by class
            unique_values, counts = np.unique(vals, return_counts=True)
            class_counts = dict(zip(unique_values, counts))
            # print(f"class counts: {class_counts}")

            # calculate percent of total area occupied by each class
            total = np.sum(counts)
            class_percents = {k: v / total for k, v in class_counts.items()}
            print(f"class percents: {class_percents}")

            # 2. Create a colormap for the raster data.
            # Filter out nodata values from unique_values for colormap
            unique_values_filtered = [
                val
                for val in unique_values
                if (nodata is None or val != nodata)
            ]
            num_classes = len(unique_values_filtered)
            colors = plt.cm.tab20(
                np.linspace(0, 1, num_classes)
            )  # Use a colormap with enough distinct colors
            cmap = plt.cm.colors.ListedColormap(colors)
            bounds = (
                np.array(
                    unique_values_filtered + [unique_values_filtered[-1] + 1]
                )
                - 0.5
            )
            norm = plt.cm.colors.BoundaryNorm(bounds, cmap.N)

            # 3. Plot the clipped raster data
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
            # Pass the buffer bounds directly to the extent argument of rasterio.plot.show
            extent = buffer_reproj.bounds[0]
            rasterio.plot.show(
                band,
                transform=transform,
                ax=ax1,
                cmap=cmap,
                norm=norm,
                extent=extent,
            )

            # 4. Plot the buffer geometry - convert to GeoSeries for plotting
            gpd.GeoSeries(buffer_reproj).plot(
                ax=ax1, facecolor="none", edgecolor="red", linewidth=2
            )

            # 5. Add a title to the plot
            ax1.set_title(
                f"Land Cover for Location {row['site_id']}, Buffer {buffer_size}m, Year {year}"
            )

            # 6. Create a pandas DataFrame from the class_counts and class_percents dictionaries.
            results_df = pd.DataFrame(
                {
                    "count": pd.Series(class_counts),
                    "percent": pd.Series(class_percents),
                }
            )

            # 7. Add a column to the DataFrame for the NLCD class names
            results_df["class_name"] = results_df.index.map(NLCD_CLASS_LUT)

            # 8. Generate a pie chart
            ax2.pie(
                results_df["percent"],
                labels=results_df["class_name"],
                autopct="%1.1f%%",
                startangle=90,
                colors=colors[: len(results_df)],
            )

            # 9. Add a title to the pie chart.
            ax2.set_title(
                f"Land Cover Distribution for Location {row['site_id']}, Buffer {buffer_size}m, Year {year}"
            )

            # 10. Ensure both plots are displayed
            plt.show()

    return None

## Main loop


### dev loop

In [16]:
%%script false
# store results
results = []

# subset for testing
df = cwmp_df.head()

# Iterate over CWMP locations
for index, row in df.iterrows():
    print(f"Processing location {index + 1} of {len(df)}")
    # create point
    point = gpd.points_from_xy(
        [row["longitude"]], [row["latitude"]], crs="EPSG:4326"
    )

    # iterate over buffer distances
    for buffer in buffers[:1]:
        print(f"Processing buffer {buffer}m")
        # iterate over years
        for year in years[:1]:
            print(f"Processing year {year}")

            # process nlcd data for row and append to results
            results.append(process_one_landcover(point, buffer, year, "nlcd"))

            # process nalcms data for row and append to results
            results.append(process_one_landcover(point, buffer, year, "nalcms"))

            # process nwi data for row and append to results
            results.append(process_one_nwi(point, buffer, "nwi"))

            # process a row and plot it
            # process_one_for_show(row, buffer, year)

# merge all dicts in `results` into one flat dict
combined = {k: v for d in results for k, v in d.items()}

# now this matches the single-row index of df
new_cols = pd.DataFrame([combined], index=df.index)
results_df = df.join(new_cols)
display(results_df)

Couldn't find program: 'false'


### optimized loop

In [17]:
%%script false
from tqdm.auto import tqdm

rows_out = []  # one dict per input row
df = cwmp_df.head()  # full dataframe
loc_iter = df.itertuples(index=True)  # keep original index
for row in tqdm(
    loc_iter,
    total=len(df),
    desc="Locations",
    position=0,
    dynamic_ncols=True,
):
    row_results = []  # collect dicts for this row
    point = gpd.points_from_xy([row.longitude], [row.latitude], crs="EPSG:4326")

    with tqdm(
        total=len(buffers), desc="Buffers", leave=False, position=1
    ) as p_buf:
        for b_idx, buffer in enumerate(buffers, 1):
            p_buf.set_postfix(buffer=buffer, b_num=b_idx)

            with tqdm(
                total=len(years), desc="Years", leave=False, position=2
            ) as p_year:
                for y_idx, year in enumerate(years, 1):
                    # show both buffer + year on the inner bar
                    p_year.set_postfix(buffer=buffer, year=year, y_num=y_idx)
                    row_results.append(
                        process_one_landcover(point, buffer, year, "nlcd")
                    )
                    row_results.append(
                        process_one_landcover(point, buffer, year, "nalcms")
                    )
                    row_results.append(process_one_nwi(point, buffer, "nwi"))
                    
                    p_year.update(1)
            p_buf.update(1)

    # merge this location's dicts into one dict
    merged = {k: v for d in row_results for k, v in d.items()}
    merged["_source_index"] = row.Index  # keep the original row index
    rows_out.append(merged)

# build a DataFrame aligned to original df
new_cols = pd.DataFrame(rows_out).set_index("_source_index").reindex(df.index)
results_df = df.join(new_cols)

display(results_df.head())

Couldn't find program: 'false'


### parallelized

In [ ]:
# Robust, notebook/Windows-safe parallelism with joblib (loky) + tqdm,
# avoiding closures and unpicklable objects. Adds per-call diagnostics.

import os, time, traceback
from typing import List, Dict, Tuple, Optional
import pandas as pd
import geopandas as gpd
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# ── limit nested threading to prevent oversubscription ──
os.environ.setdefault("GDAL_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# ── tqdm <-> joblib bridge without closures ──
from joblib.parallel import BatchCompletionCallBack


class _TqdmBatchCompletionCallback(BatchCompletionCallBack):
    def __init__(self, *args, **kwargs):
        self._pbar = kwargs.pop("pbar", None)
        super().__init__(*args, **kwargs)

    def __call__(self, *args, **kwargs):
        if self._pbar is not None:
            self._pbar.update(n=self.batch_size)
        return super().__call__(*args, **kwargs)


def _monkeypatch_tqdm_joblib(pbar):
    """Context manager to show joblib progress in tqdm without capturing closures."""
    from contextlib import contextmanager

    @contextmanager
    def _ctx():
        orig = Parallel._backend_callback

        def _factory(*a, **k):
            k["pbar"] = pbar
            return _TqdmBatchCompletionCallback(*a, **k)

        Parallel._backend_callback = _factory
        try:
            yield
        finally:
            Parallel._backend_callback = orig

    return _ctx()


# ── minimal row iterator ──
def _iter_min_rows(df: pd.DataFrame):
    for r in df.itertuples(index=True):
        yield int(r.Index), float(r.longitude), float(r.latitude)


# ── single-call wrapper with timing + error capture (NO closures) ──
def _call_with_diag(
    tag: str, func_name: str, *args, **kwargs
) -> Tuple[Optional[dict], Dict]:
    """
    func_name: name of a global callable available in the worker's module scope
               (e.g., 'process_one_landcover' or 'process_one_nwi')
    """
    t0 = time.time()
    ok, err, tb, out = True, None, None, None
    try:
        fn = globals()[
            func_name
        ]  # resolve in child without pickling the callable
        out = fn(*args, **kwargs)
    except Exception as e:
        ok, err, tb = False, repr(e), traceback.format_exc(limit=8)
    diag = {
        "tag": tag,
        "ok": ok,
        "elapsed_sec": round(time.time() - t0, 4),
        "error": err,
        "traceback": tb,
    }
    return (out if ok else None, diag)


# ── top-level worker (importable, no captured state) ──
def compute_one_row_instrumented(
    row_index: int, lon: float, lat: float, buffers: List, years: List
) -> Dict:
    point = gpd.points_from_xy([lon], [lat], crs="EPSG:4326")

    merged_parts: List[dict] = []
    diags: List[Dict] = []

    for buffer in buffers:
        for year in years:
            # NLCD
            res, d = _call_with_diag(
                f"nlcd|b={buffer}|y={year}",
                "process_one_landcover",
                point,
                buffer,
                year,
                "nlcd",
            )
            d["row_index"] = row_index
            diags.append(d)
            if res is not None:
                merged_parts.append(res)

            # NALCMS
            res, d = _call_with_diag(
                f"nalcms|b={buffer}|y={year}",
                "process_one_landcover",
                point,
                buffer,
                year,
                "nalcms",
            )
            d["row_index"] = row_index
            diags.append(d)
            if res is not None:
                merged_parts.append(res)

            # NWI
            res, d = _call_with_diag(
                f"nwi|b={buffer}", "process_one_nwi", point, buffer, "nwi"
            )
            d["row_index"] = row_index
            diags.append(d)
            if res is not None:
                merged_parts.append(res)

    merged = {k: v for dct in merged_parts for k, v in dct.items()}
    merged["_source_index"] = row_index
    return {"merged": merged, "diags": diags}


# ── main driver ──
def run_parallel_locations_joblib_instrumented(
    df: pd.DataFrame,
    buffers: List,
    years: List,
    n_jobs: int = 4,
    diag_csv: Optional[str] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    jobs = list(_iter_min_rows(df))
    if not jobs:
        return df.copy(), pd.DataFrame(
            columns=[
                "row_index",
                "tag",
                "ok",
                "elapsed_sec",
                "error",
                "traceback",
            ]
        )

    # batch_size=1 → steady progress + early failure surfacing
    with tqdm(
        total=len(jobs), desc="Locations (done)", dynamic_ncols=True
    ) as pbar, _monkeypatch_tqdm_joblib(pbar):
        results = Parallel(
            n_jobs=n_jobs,
            prefer="processes",
            batch_size=1,
            verbose=0,
        )(
            [
                delayed(compute_one_row_instrumented)(
                    idx, lon, lat, buffers, years
                )
                for (idx, lon, lat) in jobs
            ]
        )

    merged_rows = [r["merged"] for r in results]
    all_diags = [d for r in results for d in r["diags"]]

    new_cols = (
        pd.DataFrame(merged_rows).set_index("_source_index").reindex(df.index)
    )
    results_df = df.join(new_cols)

    diag_df = pd.DataFrame(all_diags)
    if diag_csv:
        diag_df.to_csv(diag_csv, index=False)

    return results_df, diag_df


# ── usage ──
df = cwmp_df.head()
results_df, diag_df = run_parallel_locations_joblib_instrumented(
    df, buffers=buffers, years=years, n_jobs=4, diag_csv="diag_calls.csv"
)
display(results_df.head())
display(diag_df.head())

Locations (done):   0%|          | 0/5 [00:00<?, ?it/s]

AttributeError: type object 'Parallel' has no attribute '_backend_callback'

: 

In [21]:
# in your current notebook
from joblib import Parallel, delayed
import time

res = Parallel(n_jobs=4, prefer="processes")(
    delayed(lambda i: (i, time.sleep(0.2)))(i) for i in range(10)
)
print("ok", len(res))

ok 10


In [ ]:
# export results to csv
outpath = r"C:\Projects\isabel_bird\data\output\cwmp_bird_count_landcover_and_wetland_summary.csv"
results_df.to_csv(outpath, index=False)